# Lewisham · `valid_date` (web: "Valid date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and values that are not dates

Fetch `valid_date` for every Lewisham application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it;
- **not a date**: a value that doesn't parse as `DD/MM/YYYY`.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "valid_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "valid_date"])
print("Lewisham applications:", len(df))

Lewisham applications: 29021


In [3]:
value = df["valid_date"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
parsed = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")
not_a_date = value.notna() & ~is_blank & parsed.isna()

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "not a date": int(not_a_date.sum()),
    "valid DD/MM/YYYY date": int(parsed.notna().sum()),
})
summary["total"] = summary.sum()
display(summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)}))

print("Values that are not dates:")
df[not_a_date]

,count,share_%
null,4888,16.84
empty or blank,0,0.00
not a date,0,0.00
valid DD/MM/YYYY date,24133,83.16
total,29021,100.00


Values that are not dates:


,id,valid_date


### Findings · Step 1

Live API, 2026-11-11, 29,021 Lewisham records:

| | Count | Share |
|---|---|---|
| null | 4,888 | 16.84% |
| empty or blank | 0 | 0% |
| not a date | 0 | 0% |
| valid `DD/MM/YYYY` date | 24,133 | 83.16% |

- **Clean format:** every filled value is a real `DD/MM/YYYY` date.
- **4,888 have no valid_date** (Brent: 3,928). Lewisham has 4,312 legacy records
  (target-date notebook, Step 2) and 1,151 empty records; how they overlap with the 4,888
  is not checked yet.

## Step 2 · Records with no valid_date: legacy, empty, or other?

Fetch `bo_system` and the four fields of the empty-record rule (`status`, `decision`,
`decision_date`, `description`) for every record. A record is **legacy** when `bo_system`
is FME Legacy Data Import, and **empty** when the four fields are all empty (the
`drop_empty_records` condition; empty text counts as empty).

In [4]:
COLS = ["id", "valid_date", "bo_system", "status", "decision", "decision_date", "description"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

full = pd.DataFrame(rows).reindex(columns=COLS)


def is_empty(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip() == "")


full["no_valid_date"] = is_empty(full["valid_date"])
full["legacy"] = full["bo_system"] == "FME Legacy Data Import"
full["empty_record"] = pd.concat([is_empty(full[f]) for f in ["status", "decision", "decision_date", "description"]], axis=1).all(axis=1)

print("No valid_date:", int(full["no_valid_date"].sum()))
print()
print(pd.crosstab([full["legacy"], full["empty_record"]], full["no_valid_date"], margins=True).to_string())
print()
print("No valid_date, by bo_system:")
print(full.loc[full["no_valid_date"], "bo_system"].value_counts(dropna=False).to_string())

No valid_date: 4888

no_valid_date        False  True    All
legacy empty_record                    
False  False         23509    49  23558
       True            624   527   1151
True   False             0  4312   4312
All                  24133  4888  29021

No valid_date, by bo_system:
bo_system
FME Legacy Data Import    4312
CSV Data Import            552
Portal                      24


In [5]:
groups = full[full["no_valid_date"]].assign(
    group=lambda t: t["legacy"].map({True: "legacy", False: ""}) + t["empty_record"].map({True: "empty", False: ""}))
groups["group"] = groups["group"].replace("", "other")
print(pd.crosstab(groups["group"], groups["bo_system"].fillna("(none)"), margins=True).to_string())
print()
print("Empty records WITH a valid_date, by bo_system:",
      full.loc[full["empty_record"] & ~full["no_valid_date"], "bo_system"].value_counts().to_dict())

bo_system  CSV Data Import  FME Legacy Data Import  Portal   All
group                                                           
empty                  527                       0       0   527
legacy                   0                    4312       0  4312
other                   25                       0      24    49
All                    552                    4312      24  4888

Empty records WITH a valid_date, by bo_system: {'CSV Data Import': 624}


### Findings · Step 2

The 4,888 records with no valid_date:

| group | Count | bo_system |
|---|---|---|
| legacy | 4,312 | FME Legacy Data Import |
| empty record (dropped at load) | 527 | CSV Data Import |
| other | 49 | CSV Data Import 25, Portal 24 |

- **Legacy records never have a valid_date:** all 4,312, as in Brent.
- **Empty records split:** 527 have no valid_date, but **624 have one.** So an "empty"
  record can still carry a valid date (like DC/24/137429, status Step 3). They are dropped
  either way.
- **49 real gaps** after legacy and empty records: 25 from the CSV import, 24 of the 33
  `Portal` records. Not looked at yet.
- After cleaning: 4,312 legacy + 49 others = **4,361 records with no valid_date**.

## Step 3 · The 49 records with no valid_date that are neither legacy nor empty

Fetched by id with their outcome, dates, type and URL.

In [6]:
MORE = ["id", "lpa_app_no", "bo_system", "status", "decision", "decision_date", "decision_target_date",
        "application_type", "url_planning_app", "last_updated", "description"]
gap_ids = full.loc[full["no_valid_date"] & ~full["legacy"] & ~full["empty_record"], "id"].tolist()
body = {"size": len(gap_ids), "_source": MORE, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": gap_ids}}]}}}
gaps = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=MORE)
gaps["url"] = gaps.pop("url_planning_app").notna()
gaps["last_updated"] = gaps["last_updated"].str.slice(0, 10)
gaps["description"] = gaps["description"].str.slice(0, 45)
print(len(gaps), "records")
print(gaps[["bo_system", "status", "decision"]].fillna("(none)").value_counts().to_string())
print()
print(gaps.sort_values(["bo_system", "id"]).to_string(index=False))

49 records
bo_system        status     decision
Portal           Completed  Approved    21
CSV Data Import  Approved   Approved    20
Portal           Commenced  Approved     2
CSV Data Import  Lapsed     Approved     2
                 Commenced  Approved     2
Portal           Completed  Refused      1
CSV Data Import  Completed  Approved     1

                   id   lpa_app_no       bo_system    status decision decision_date decision_target_date application_type last_updated                                   description   url
 Lewisham-DC_13_83358  DC/13/83358 CSV Data Import    Lapsed Approved    10/03/2015                 None        All Other   2026-06-24 The comprehensive redevelopment of Convoys Wh False
Lewisham-DC_14_088711 DC/14/088711 CSV Data Import Commenced Approved    13/03/2015                 None        All Other   2025-03-20 The reconfiguration of the ground and basemen False
Lewisham-DC_14_089680 DC/14/089680 CSV Data Import Commenced Approved    07/04/2015      

### Findings · Step 3

- **All 49 are decided** (48 Approved, 1 Refused) and **none has a URL or a target date.**
- **25 from the CSV import:**
  - 20 are Prior Approval applications, approved 2022–2025, status Approved, all
    last_updated 2026-09-22 (one batch). One of them, DC/24/134931, is described as
    "Application for outline planning permission".
  - 5 older approvals (2015–2021) whose status moved on (Lapsed 2, Commenced 2,
    Completed 1).
- **24 `Portal` records** (of 33): all status Completed (22) or Commenced (2), decided
  2009–2024; 8 are Prior Approvals. Older references use the legacy style (DC08/69448,
  DC10/75331, …).
- **A pattern, not random gaps:** the missing valid_date goes with a missing URL and
  target date, and with Prior Approvals (28 of 49). Why is not known.

## Step 4 · valid_date after decision_date

An application is validated before it is decided, so `valid_date` after `decision_date`
cannot be right. Lewisham-95_038596 (status Historical) has a 1996 decision and a 2023
valid date that the portal ties to a migration. Count every such record, by how far apart
the dates are, and show when those valid dates fall.

In [7]:
full["valid"] = pd.to_datetime(full["valid_date"], format="%d/%m/%Y", errors="coerce")
full["decided"] = pd.to_datetime(full["decision_date"], format="%d/%m/%Y", errors="coerce")
both = full.dropna(subset=["valid", "decided"]).copy()
both["days"] = (both["decided"] - both["valid"]).dt.days
after = both[both["days"] < 0].copy()
print("Records with both dates:", len(both), "| valid_date after decision_date:", len(after))
print()
gap = -after["days"]
after["gap"] = pd.cut(gap, bins=[0, 30, 365, 3650, 100000], labels=["1-30 days", "31 days-1 year", "1-10 years", "over 10 years"])
print("How far after the decision:")
print(after["gap"].value_counts(sort=False).to_string())
print()
print("By bo_system:", after["bo_system"].value_counts().to_dict())
print("By status:", after["status"].value_counts().to_dict())

Records with both dates: 22935 | valid_date after decision_date: 15

How far after the decision:
gap
1-30 days         4
31 days-1 year    7
1-10 years        2
over 10 years     2

By bo_system: {'CSV Data Import': 15}
By status: {'Withdrawn': 7, 'Approved': 3, 'Refused': 2, 'Historical': 1, 'No Objection to Proposal (OBS only)': 1}


In [8]:
# The records more than a year apart: when do their valid dates fall?
far = after[after["days"] < -365].sort_values("valid")
print(len(far), "records more than a year apart")
print(far["valid"].dt.to_period("M").value_counts().sort_index().to_string())
print()
print(far[["id", "status", "decision", "decision_date", "valid_date", "bo_system"]].to_string(index=False))

4 records more than a year apart
valid
2023-08    1
2023-11    1
2025-12    1
2026-08    1
Freq: M

                     id     status decision decision_date valid_date       bo_system
  Lewisham-DC_21_122222   Approved Approved    06/08/2021 14/08/2023 CSV Data Import
     Lewisham-95_038596 Historical  Refused    30/10/1996 28/11/2023 CSV Data Import
Lewisham-DC_13_082322_X   Approved Approved    10/04/2013 11/12/2025 CSV Data Import
Lewisham-TEST_22_022222   Approved Approved    06/03/2024 04/08/2026 CSV Data Import


In [9]:
# The 11 within a year.
near = after[after["days"] >= -365].sort_values("days")
print(near[["id", "status", "decision", "decision_date", "valid_date", "days"]].to_string(index=False))

                   id                              status                            decision decision_date valid_date  days
Lewisham-DC_23_132806                           Withdrawn                           Withdrawn    27/11/2023 03/10/2024  -311
Lewisham-DC_22_129284                           Withdrawn                           Withdrawn    27/01/2023 23/11/2023  -300
Lewisham-DC_20_116858                             Refused                             Refused    07/01/2020 17/06/2020  -162
Lewisham-DC_25_140274                           Withdrawn                           Withdrawn    10/07/2025 06/11/2025  -119
Lewisham-DC_23_131379                           Withdrawn                           Withdrawn    22/06/2023 17/10/2023  -117
Lewisham-DC_19_114639                           Withdrawn                           Withdrawn    27/11/2019 20/01/2020   -54
Lewisham-DC_19_115117                             Refused                             Refused    15/01/2020 21/02/2020   -37


### Findings · Step 4

**15 of 22,935** records with both dates have valid_date after decision_date, all from
the CSV import:

- **Migration is not a pattern:** only 4 are more than a year apart:
  - Lewisham-95_038596 (Historical, decided 1996, valid 2023): the migrated record
    already known.
  - Lewisham-DC_13_082322_X (Approved 2013, valid 11/12/2025): the `_X` suffix is also
    on two TEST records; not checked.
  - Lewisham-DC_21_122222 (Approved 2021, valid 2023): not explained.
  - Lewisham-TEST_22_022222: a test record, dropped at load.
- **7 of the 11 within a year are Withdrawn:** withdrawn before or around validation, so
  the decision_date may be the withdrawal date. Not confirmed.
- The other 4 within a year: 2 Refused (37 and 162 days), 1 No Objection (1 day), and
  DC/26/143311 (5 days; the portal shows the same dates, status Unknown).
- **For the bot:** after cleaning, 14 records (0.06%) have this order. A decision time
  (valid_date → decision_date) for them would be negative.

## Step 5 · Type check and most common values

The Python type of every value in `df` (Step 1), since aggregations hide the JSON type.
Then the most common dates, the range, and the count by year.

In [10]:
types = df["valid_date"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Types:", types.value_counts().to_dict())

filled = df["valid_date"].dropna()
dates = pd.to_datetime(filled, format="%d/%m/%Y")
print("Earliest:", dates.min().date(), "| latest:", dates.max().date(),
      "| after today (2026-11-11):", int((dates > "2026-11-11").sum()))
print()
print("Most common valid dates:")
print(filled.value_counts().head(10).to_string())
print()
print("By year:")
print(dates.dt.year.value_counts().sort_index().to_string())

Types: {'str': 24133, 'NoneType': 4888}


Earliest: 2010-09-14 | latest: 2026-11-06 | after today (2026-11-11): 0

Most common valid dates:
valid_date
15/12/2022    43
22/06/2026    42
15/03/2022    35
19/04/2021    33
17/06/2019    33
18/01/2021    33
06/07/2020    32
15/02/2021    32
25/02/2019    31
03/06/2019    31

By year:
valid_date
2010       1
2014       1
2016       2
2017       5
2018       6
2019    3387
2020    3062
2021    4213
2022    3514
2023    2951
2024    2608
2025    2383
2026    2000


### Findings · Step 5

- **Strings or null only** (24,133 strings, 4,888 null); no numbers. Every string is a valid
  `DD/MM/YYYY` date (Step 1), from 2010-09-14 to 2026-11-06; none after today.
- **No placeholder date:** the most common valid date (15/12/2022) has 43 records.
- **Valid dates start in 2019:** only 15 records have a valid_date before 2019 (2010–2018);
  from 2019 on, 2,000–4,213 a year (most in 2021). Legacy records never have a valid_date
  (Step 2); how many records from before 2019 are legacy is not counted here.
- **So counts by submission year cover 2019 on only.** Brent's valid dates go back
  further; not compared here.